In [ ]:

Programming Stack
•	Language: Python
•	Libraries:
o	NumPy
o	Pandas
o	Scikit-learn
o	XGBoost
o	TensorFlow / PyTorch
o	GeoPandas
o	PySAL


#Data Preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load data
df = pd.read_csv("nigeria_weather.csv")

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort for time series
df = df.sort_values('Date')

# Handle missing values
df.interpolate(method='linear', inplace=True)

# Features
features = ['Temp_Max', 'Temp_Min', 'Humidity', 'Latitude', 'Longitude']
target = 'Rainfall'

X = df[features]
y = df[target]

# Train-test split (chronological)
split = int(len(df) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Random Forest Model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=3,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Metrics
mae_rf = mean_absolute_error(y_test, rf_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))
r2_rf = r2_score(y_test, rf_pred)

XGBoost Model

from xgboost import XGBRegressor

xgb = XGBRegressor(
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    n_estimators=300
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

# Metrics
mae_xgb = mean_absolute_error(y_test, xgb_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, xgb_pred))
r2_xgb = r2_score(y_test, xgb_pred)

#LSTM Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Reshape for LSTM [samples, timesteps, features]
X_train_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

model.fit(X_train_lstm, y_train,
          epochs=100,
          batch_size=32,
          validation_split=0.2,
          verbose=1)

lstm_pred = model.predict(X_test_lstm)

#Bi-LSTM Model

from tensorflow.keras.layers import Bidirectional

model_bi = Sequential([
    Bidirectional(LSTM(64), input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(1)
])

model_bi.compile(optimizer='adam', loss='mse')

model_bi.fit(X_train_lstm, y_train,
             epochs=100,
             batch_size=32,
             validation_split=0.2)

bilstm_pred = model_bi.predict(X_test_lstm)



#Transformer Model
import tensorflow as tf
from tensorflow.keras.layers import Dense, LayerNormalization, MultiHeadAttention

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.norm1 = LayerNormalization()
        self.ffn = tf.keras.Sequential([
            Dense(64, activation="relu"),
            Dense(embed_dim)
        ])
        self.norm2 = LayerNormalization()

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.norm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.norm2(out1 + ffn_output)

# Model
inputs = tf.keras.Input(shape=(1, X_train.shape[1]))
x = TransformerBlock(embed_dim=X_train.shape[1], num_heads=4)(inputs)
x = Dense(1)(x)

model_trans = tf.keras.Model(inputs, x)
model_trans.compile(optimizer='adam', loss='mse')

model_trans.fit(X_train_lstm, y_train,
                epochs=100,
                batch_size=32)

trans_pred = model_trans.predict(X_test_lstm)

Moran’s I (Spatial Analysis)
import pysal.lib
from esda.moran import Moran

# Example spatial weights
coords = list(zip(df['Latitude'], df['Longitude']))
w = pysal.lib.weights.KNN.from_array(coords, k=5)

mi = Moran(df['Rainfall'], w)

print("Moran's I:", mi.I)
print("p-value:", mi.p_sim)

Evaluation Metrics
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2
